# Encoder Comparison: Basis vs Binary vs Trig

This notebook compares the three encoders provided by `MPSFast.Encoders`
on the same Heston path dataset:

| Encoder | Sites/timestep | Physical dim `d` | Feature map |
|---------|---------------|-----------------|-------------|
| `BasisEncoder(m)` | 1 | `2^m` | None (one-hot) |
| `BinaryEncoder(m)` | `m` | 2 | None (one-hot) |
| `TrigEncoder(m, d_feat)` | 1 | `d_feat` | `K×d_feat` trig harmonics |

We compare: NLL convergence speed, bond dimensions needed, sampling fidelity, and option prices.

In [11]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "../.."))
Pkg.resolve()
Pkg.instantiate()

using MPSFast
using MPSFast.Encoders
using Random, LinearAlgebra, Statistics, Printf, Distributions

  Activating project at `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl`
     Project No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Project.toml`
    Manifest No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Manifest.toml`


## 1. Generate Heston Paths

In [12]:
function simulate_heston(N, M; S0=100.0, V0=0.04, κ=2.0, θ=0.04,
                          ξ=0.3, ρ=-0.7, r=0.0, dt=1/252, rng=Random.default_rng())
    S = Matrix{Float64}(undef, N, M)
    for i in 1:N
        s = S0; v = V0
        for t in 1:M
            z1 = randn(rng); z2 = randn(rng)
            w1 = z1
            w2 = ρ * z1 + sqrt(1 - ρ^2) * z2
            v  = max(v + κ*(θ-v)*dt + ξ*sqrt(max(v,0.0)*dt)*w2, 0.0)
            s  = s * exp((r-0.5*v)*dt + sqrt(max(v,0.0)*dt)*w1)
            S[i, t] = s
        end
    end
    return S
end

rng = MersenneTwister(2024)
N_train, M = 5_000, 20
paths = simulate_heston(N_train, M; rng=rng)
println("Training paths: ", size(paths))

Training paths: (5000, 20)


## 2. Set Up Encoders

In [13]:
m = 4    # 2^4 = 16 buckets

enc_basis  = BasisEncoder(m)
enc_binary = BinaryEncoder(m)
enc_trig   = TrigEncoder(m, 8; angle_mode = :discrete_cell_pi)

for enc in (enc_basis, enc_binary, enc_trig)
    fit_grid!(enc, paths)
end

xi_basis  = encode_paths(enc_basis,  paths)
xi_binary = encode_paths(enc_binary, paths)
xi_trig   = encode_paths(enc_trig,   paths)  # same bucket indices as basis

D_max = 64
println("\nEncoder overview:")
for (name, enc) in [("Basis", enc_basis), ("Binary", enc_binary), ("Trig", enc_trig)]
    Ml = chain_length(enc, M)
    d  = site_dim(enc)
    println("  ", rpad(name, 8), "  chain_length=", Ml, "  site_dim=", d,
            "  feature_map=", feature_map(enc) === nothing ? "nothing" : "$(size(feature_map(enc)))")
end


Encoder overview:
  Basis     chain_length=20  site_dim=16  feature_map=nothing
  Binary    chain_length=80  site_dim=2  feature_map=nothing
  Trig      chain_length=20  site_dim=8  feature_map=(16, 8)


## 3. Train Each Encoder

In [14]:
n_epochs = 25
η        = 5e-4
ε_cut    = 1e-5
rng_init = MersenneTwister(7)

# BasisEncoder
mps_basis = init_mps(chain_length(enc_basis, M), site_dim(enc_basis), D_max; rng=rng_init)
nll_basis = train_mps!(mps_basis, xi_basis, n_epochs, η, D_max, ε_cut;
                       verbose=false, nll_samples=500)
println("Basis  final NLL: ", round(nll_basis[end],  digits=4))

Basis  final NLL: 14.1922


In [15]:
# BinaryEncoder
mps_binary = init_mps(chain_length(enc_binary, M), site_dim(enc_binary), D_max; rng=rng_init)
nll_binary = train_mps!(mps_binary, xi_binary, n_epochs, η, D_max, ε_cut;
                        verbose=false, nll_samples=500)
println("Binary final NLL: ", round(nll_binary[end], digits=4))

Binary final NLL: 16.1728


In [16]:
# TrigEncoder — uses Gram-weighted training
Phi_trig  = Float32.(feature_map(enc_trig))
mps_trig  = init_mps(chain_length(enc_trig, M), site_dim(enc_trig), D_max; rng=rng_init)
nll_trig  = train_mps!(mps_trig, xi_trig, n_epochs, η, D_max, ε_cut;
                       feature_phi = Phi_trig, verbose=false, nll_samples=500)
println("Trig   final NLL: ", round(nll_trig[end],   digits=4))

Trig   final NLL: 28.5779


In [17]:
# NLL convergence table
println("\nEpoch | Basis NLL | Binary NLL | Trig NLL")
for e in 1:n_epochs
    @printf "  %3d | %9.4f | %10.4f | %8.4f\n" e nll_basis[e] nll_binary[e] nll_trig[e]
end


Epoch | Basis NLL | Binary NLL | Trig NLL
    1 |   23.0710 |    36.0256 |  30.4233
    2 |   17.5462 |    26.6859 |  29.8765
    3 |   17.2156 |    23.5640 |  29.5434
    4 |   17.0683 |    22.1605 |  29.6055
    5 |   16.7797 |    21.1191 |  29.4322
    6 |   16.3471 |    20.7748 |  29.1118
    7 |   16.4076 |    20.0870 |  29.1630
    8 |   16.2041 |    19.2932 |  29.0518
    9 |   16.0577 |    19.2906 |  29.0290
   10 |   15.6332 |    18.9060 |  28.9720
   11 |   15.6727 |    18.1483 |  28.9346
   12 |   15.2675 |    17.8632 |  28.8852
   13 |   15.1627 |    18.0505 |  28.8221
   14 |   15.1977 |    17.5238 |  28.7833
   15 |   14.8435 |    17.6574 |  28.8494
   16 |   14.6020 |    17.2448 |  28.7847
   17 |   14.7350 |    16.9122 |  28.7676
   18 |   14.6191 |    16.8987 |  28.7867
   19 |   14.4665 |    17.0192 |  28.5734
   20 |   14.5204 |    16.7250 |  28.6605
   21 |   14.3280 |    16.4795 |  28.5002
   22 |   14.3203 |    16.5187 |  28.5746
   23 |   14.2075 |    16.4722 | 

## 4. Bond Dimension Analysis

In [18]:
function bond_profile(mps)
    [size(mps[j], 3) for j in 1:length(mps)-1]
end

println("Bond dimensions:")
println("  Basis : ", bond_profile(mps_basis))
println("  Binary: ", bond_profile(mps_binary))
println("  Trig  : ", bond_profile(mps_trig))

# Entanglement entropy profiles
_, entr_basis  = bipartite_entropies(mps_basis)
_, entr_binary = bipartite_entropies(mps_binary)
_, entr_trig   = bipartite_entropies(mps_trig)

println("\nBipartite entropies (nats):")
println("  Basis  (max): ", round(maximum(filter(isfinite, entr_basis)),  digits=4))
println("  Binary (max): ", round(maximum(filter(isfinite, entr_binary)), digits=4))
println("  Trig   (max): ", round(maximum(filter(isfinite, entr_trig)),   digits=4))

Bond dimensions:
  Basis : [16, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 16]
  Binary: [2, 4, 8, 16, 32, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 32, 16, 8, 4, 2]
  Trig  : [8, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 8]

Bipartite entropies (nats):
  Basis  (max): 3.2208
  Binary (max): 2.7236
  Trig   (max): 3.5906


## 5. Sampling Fidelity

In [19]:
N_samp = 2_000

samp_basis,  _ = sample_paths(enc_basis,  mps_basis,  N_samp; seed=1)
samp_binary, _ = sample_paths(enc_binary, mps_binary, N_samp; seed=1)
samp_trig,   _ = sample_paths(enc_trig,   mps_trig,   N_samp; seed=1)

# Compare terminal distribution statistics
t_last = M
println("Terminal price S[M] statistics (t=", t_last, "):")
println("  Source       Mean      Std       Skew")
function stats(v)
    μ = mean(v); σ = std(v)
    sk = mean(((v .- μ)./σ).^3)
    (μ, σ, sk)
end
for (name, samp) in [("Training", paths), ("Basis", samp_basis),
                     ("Binary", samp_binary), ("Trig", samp_trig)]
    μ, σ, sk = stats(samp[:, t_last])
    @printf "  %-10s  %6.2f    %6.2f    %5.3f\n" name μ σ sk
end

Terminal price S[M] statistics (t=20):
  Source       Mean      Std       Skew
  Training     99.16      5.57    -0.320
  Basis        97.57      5.85    -0.367
  Binary       99.39      6.97    -0.738
  Trig         97.25      7.91    -0.500


## 6. Option Pricing Comparison

In [20]:
S0      = 100.0
K_atm   = 100.0
T       = M / 252

function eu_call_price(paths, K, r=0.0)
    T_loc = size(paths, 2) / 252
    exp(-r * T_loc) * mean(max.(paths[:, end] .- K, 0.0))
end

println("European ATM Call (K=", K_atm, "):")
for (name, samp) in [("Training", paths), ("Basis", samp_basis),
                     ("Binary", samp_binary), ("Trig", samp_trig)]
    p = eu_call_price(samp, K_atm)
    @printf "  %-10s  %.4f\n" name p
end

European ATM Call (K=100.0):
  Training    1.7988
  Basis       1.2441
  Binary      2.3469
  Trig        1.7760


## 7. Summary and Recommendations

| Aspect | BasisEncoder | BinaryEncoder | TrigEncoder |
|--------|-------------|--------------|-------------|
| Chain length | `M` | `M·m` | `M` |
| Per-bond cost | `O(D³d²)` | `O(D³)` | `O(D³d²)` with `d = d_feat ≪ 2^m` |
| Bond dimension needed | high | medium | low |
| Interpretability | high | medium | requires Φ inspection |
| Recommended when | `m ≤ 4`, small `M` | `m ≥ 5`, large `M` | want compact MPS with good NLL |

**Rule of thumb:** Start with `BasisEncoder(4)` for quick experiments.
Use `TrigEncoder(4, 8)` for production to reduce bond dimension.